<h1 style="color:red;">III.Analyse IMG/IVG </h1>


<h1 style="color:red;">III.3 Les variables ayant un impact sur le fait d’etre dans le groupe IMG plutot que IVG, les violences ont encore un impact?</h1>

**Pour débuter la création du premier modèle de régression logistique, le choix des variables s’est appuyé sur tes hypothèses ainsi que sur les analyses préalables, notamment la matrice de corrélation. Les variables les plus importantes ont ainsi été retenues.**

<h4 style="color:red;"> 1.Premier modèle: Variables incluses : violences passées, violences actuelles, âge, gestité, parité, enfant à charge, profession, statut conjugal, lieu de vie. </h4>

In [14]:
df_ivg = df_ivg.rename(columns={
    'ivg_no_inclus': 'patient_id',
    'ivg_age': 'age',
    'parite_ivg': 'parite',
    'enfant_charge_ivg': 'enfant_charge',
    'profession_tot_ivg': 'profession_simpl',
    'statut_conj_simpl_ivg': 'statut_conj',
    'lieuvie_tot_ivg': 'lieuvie_total',
    'atcd_violence_ivg': 'violences_passees',
    'violences_actuelles_ivg': 'violences_actuelles',
    'gestite_ivg':'gestite'
})

df_img = df_img.rename(columns={
    'no_inclus': 'patient_id',
    'age_img': 'age',
    'parite_img': 'parite',
    'enfant_charge_img': 'enfant_charge',
    'profession_simpl_img': 'profession_simpl',
    'Statut_conj_simplifié_img': 'statut_conj',
    'lieuvie_tot_img': 'lieuvie_total',
    'atcd_violence_img': 'violences_passees',
    'violences_actuelles_img': 'violences_actuelles',
    'gestite_img':'gestite'
})

# Création de la variable cible : 0 = IVG, 1 = IMG
df_ivg['type_interruption'] = 0
df_img['type_interruption'] = 1

# Suppression des doublons
df_ivg = df_ivg.drop_duplicates(subset='patient_id')
df_img = df_img.drop_duplicates(subset='patient_id')

# Fusion
df = pd.concat([df_ivg, df_img], ignore_index=True)

# Remplissage des NaN pour les violences
df['violences_passees'] = df['violences_passees'].fillna(0).astype(int)
df['violences_actuelles'] = df['violences_actuelles'].fillna(0).astype(int)

# Suppression des lignes sans valeurs pour les autres variables
df = df.dropna(subset=['age', 'parite', 'enfant_charge','gestite', 
                       'profession_simpl', 'statut_conj', 'lieuvie_total'])

#Modèle Logit
X = df[['violences_passees', 'violences_actuelles', 'age', 'gestite',
        'parite', 'enfant_charge', 'profession_simpl', 
        'statut_conj', 'lieuvie_total']]
y = df['type_interruption']

# Ajout de l’intercept
X = sm.add_constant(X)

# Ajustement du modèle
model = sm.Logit(y, X).fit()

# Affichage du résumé
print(model.summary2())

# Calcul des Odds Ratios et IC à 95 %
params = model.params
conf = model.conf_int()
or_ci = pd.DataFrame({
    'OR': np.exp(params),
    'IC_inf': np.exp(conf[0]),
    'IC_sup': np.exp(conf[1])
})
print("\n=== Odds Ratios et Intervalles de Confiance à 95% ===")
print(or_ci.round(3))

Optimization terminated successfully.
         Current function value: 0.483970
         Iterations 7
                          Results: Logit
Model:              Logit             Method:           MLE       
Dependent Variable: type_interruption Pseudo R-squared: 0.301     
Date:               2025-12-27 20:28  AIC:              201.0048  
No. Observations:   187               BIC:              233.3159  
Df Model:           9                 Log-Likelihood:   -90.502   
Df Residuals:       177               LL-Null:          -129.40   
Converged:          1.0000            LLR p-value:      4.4171e-13
No. Iterations:     7.0000            Scale:            1.0000    
------------------------------------------------------------------
                     Coef.  Std.Err.    z    P>|z|   [0.025 0.975]
------------------------------------------------------------------
const               -0.4119   1.3333 -0.3089 0.7574 -3.0252 2.2014
violences_passees   -0.0994   0.4263 -0.2332 0.8156 -

**Interprétation:**

**Violences actuelles :** coefficient = 3.2405, **p-value = 0.0000** → **très significatif**

**OR ≈ 26** → Les femmes ayant subi des violences actuelles ont 26 fois plus de chances d’être dans le groupe IMG plutôt qu’IVG, toutes choses égales par ailleurs.

**Violences passées :** coefficient = -0.0994, p-value = 0.8156 → non significatif

**OR ≈ 0.905 → Aucun impact détecté.**

Les autres variables **(âge, gestité, parité, enfant à charge, profession, statut conjugal, lieu de vie)** ne sont pas significatives (p-values > 0.05).

Même en contrôlant toutes les autres variables, les violences actuelles ont un impact très fort et significatif sur le choix de l’IMG par rapport à l’IVG. Les violences passées, elles, n’ont pas d’effet statistiquement significatif.

**Pour la construction du deuxième modèle, une procédure de sélection des variables a été appliquée. Le modèle initial incluait l’ensemble des variables explicatives du premier modèle. Les variables ont ensuite été éliminées de manière itérative en fonction de leur significativité statistique, en supprimant à chaque étape celles dont la p-valeur était la plus élevée, jusqu’à ne conserver que les variables statistiquement significatives.**

<h4 style="color:red;"> Deuxième modèle (réduit après élimination des variables non significatives) ; Variables incluses : violences actuelles, violences passées, enfant à charge, profession, statut conjugal. </h4>

In [16]:
df_ivg = df_ivg.rename(columns={
    'ivg_no_inclus': 'patient_id',
    'enfant_charge_ivg': 'enfant_charge',
    'profession_tot_ivg': 'profession_simpl',
    'statut_conj_simpl_ivg': 'statut_conj',
    'violences_actuelles_ivg': 'violences_actuelles',
    'atcd_violence_ivg': 'violences_passees'
})

df_img = df_img.rename(columns={
    'no_inclus': 'patient_id',
    'enfant_charge_img': 'enfant_charge',
    'profession_simpl_img': 'profession_simpl',
    'Statut_conj_simplifié_img': 'statut_conj',
    'violences_actuelles_img': 'violences_actuelles',
    'atcd_violence_img': 'violences_passees'
})

# Création de la variable cible : 0 = IVG, 1 = IMG
df_ivg['type_interruption'] = 0
df_img['type_interruption'] = 1

# Suppression des doublons
df_ivg = df_ivg.drop_duplicates(subset='patient_id')
df_img = df_img.drop_duplicates(subset='patient_id')

# Fusion
df = pd.concat([df_ivg, df_img], ignore_index=True)

# Remplissage des NaN pour les violences
df['violences_actuelles'] = df['violences_actuelles'].fillna(0).astype(int)
df['violences_passees'] = df['violences_passees'].fillna(0).astype(int)

# Suppression des lignes sans valeurs pour les autres variables
df = df.dropna(subset=['enfant_charge', 
                       'profession_simpl', 'statut_conj'])

# Modèle Logit
X = df[['violences_actuelles', 'violences_passees',
        'enfant_charge', 'profession_simpl', 
        'statut_conj']]
y = df['type_interruption']

# Ajout de l’intercept
X = sm.add_constant(X)

# Ajustement du modèle
model = sm.Logit(y, X).fit()

# Affichage du résumé
print(model.summary2())

# Calcul des Odds Ratios et IC à 95 %
params = model.params
conf = model.conf_int()
or_ci = pd.DataFrame({
    'OR': np.exp(params),
    'IC_inf': np.exp(conf[0]),
    'IC_sup': np.exp(conf[1])
})
print("\n=== Odds Ratios et Intervalles de Confiance à 95% ===")
print(or_ci.round(3))

Optimization terminated successfully.
         Current function value: 0.497034
         Iterations 7
                          Results: Logit
Model:               Logit             Method:           MLE       
Dependent Variable:  type_interruption Pseudo R-squared: 0.282     
Date:                2025-12-27 20:36  AIC:              198.8848  
No. Observations:    188               BIC:              218.3034  
Df Model:            5                 Log-Likelihood:   -93.442   
Df Residuals:        182               LL-Null:          -130.14   
Converged:           1.0000            LLR p-value:      2.0081e-14
No. Iterations:      7.0000            Scale:            1.0000    
-------------------------------------------------------------------
                     Coef.  Std.Err.    z    P>|z|   [0.025  0.975]
-------------------------------------------------------------------
const               -0.6528   0.8891 -0.7343 0.4628 -2.3953  1.0897
violences_actuelles  3.2811   0.6111  5.3

**Interprétation:**

**Violences actuelles :** coefficient = 3.2811,**p-value = 0.0000** → **toujours très significatif**

**Violences passees :** coefficient = 0.0711,**p-value = 0.8585** → **non significatif**

**OR ≈ 27**→ Impact encore plus clair sur le fait d’être dans le groupe IMG.

**Enfant à charge :** coefficient = -0.433, **p-value = 0.0077**→ **Significatif**

**OR ≈ 0.65** → Les femmes ayant un enfant à charge sont moins susceptibles d’être dans le groupe IMG.

**Profession et statut conjugal:** p-values ≈ 0.06–0.08 → Tendances mais non significatives à 5%.

Même dans un modèle réduit, les violences actuelles restent l’effet le plus fort et significatif. Le fait d’avoir un enfant à charge est également associé à une probabilité moindre d’IMG. La simplification du modèle n’a donc pas changé la conclusion principale.

<h4 style="color:red;">Conclusion par rapport à la question </h4>

**Impact des violences :**

Les violences actuelles ont un impact très important et significatif sur le fait d’être dans le groupe IMG.

Les violences passées n’ont aucun effet significatif.

**Modèle complet vs modèle réduit :**

Le modèle complet permet de contrôler toutes les variables, mais le modèle réduit montre de manière plus claire et interprétable que les violences actuelles sont le facteur prédominant.